# Estimación temprana de riesgo de desnutrición crónica infantil
## Notebook único del artículo — VIII Congreso Internacional de Ciencias Informáticas (ICCS)

**Fuente:** ENDI Ronda 2 (2023-2024), INEC. Microdatos públicos (ANDA-INEC).
**Autor:** Arturo Mauricio Palomino Palacios — Maestría en Ciencia de Datos y Máquinas de Aprendizaje, UTM.

Este notebook reemplaza a `01_exploracion.ipynb`, `02_base_analitica.ipynb` y `03_trayectorias.ipynb`
como fuente única para el artículo. Los dos primeros pertenecen a otro proyecto (determinantes de la
DCI, comparación Cañar/nacional) y no se usan aquí salvo por la construcción de la base transversal
`nin`, que se reconstruye desde cero más abajo con solo las variables que este artículo necesita.

**Reglas metodológicas (no reabrir sin razón documentada):**

1. Diseño muestral complejo — no se usa para estimar razones puntuales de sesgo, sensibilidad o AUC en
   este artículo (esas son propiedades de la medición y del modelo, no estimaciones poblacionales), pero
   si en algún punto se reporta una prevalencia agregada, debe ponderarse.
2. Los identificadores son *strings* y la ausencia se codifica como `''`, no `NaN`.
3. `validate=` en todo *merge*, y verificar cobertura después de cada uno.
4. Criterio OMS: se excluyen puntajes Z con \|Z\| > 6 en ambas fuentes (carné y medición ENDI).
5. **El sesgo de calibración del carné tiene dos usos distintos y no deben confundirse:**
   - Sesgo global (todas las edades, n≈4.761) → describe la calidad general del carné (Tabla 4 del artículo).
   - Sesgo restringido a ≤18 meses → es el que se aplica para corregir las mediciones que alimentan el
     modelo, porque el modelo solo usa mediciones ≤18 meses. **No es el mismo número.**
6. Toda estimación de desempeño del modelo (AUC, matriz de confusión, calibración) sale de
   `cross_val_predict` con `GroupKFold` agrupado por `id_upm`. Nunca de una reconstrucción aproximada.


## 1. Configuración y funciones auxiliares

In [1]:
import pandas as pd, numpy as np
import pyreadstat, joblib
from pathlib import Path
from scipy import stats

pd.set_option('display.width', 140)

RAIZ = Path("/home/mauricio/proyectos/endi")
F = lambda n: str(RAIZ / f"BDD_ENDI_R2_{n}.dta")
K = ['id_upm', 'id_viv', 'id_hogar', 'id_per']


def rango(s, lo, hi):
    """Numérico dentro de rango plausible; fuera de rango -> nulo.
    Descarta también los códigos de no-respuesta (8888, 9999) que vienen como valores."""
    x = pd.to_numeric(s, errors='coerce')
    return x.where((x >= lo) & (x <= hi))


def cobertura(df, cols):
    """% de filas con dato válido, tratando '' como ausente (llaves string)."""
    out = {}
    for c in cols:
        x = df[c]
        out[c] = ((x.notna()) & (x.astype(str) != '')).mean()
    return pd.Series(out).sort_values().round(3)


## 2. Base transversal (`nin`)

Solo las variables que usa este artículo: resultado (`dcronica2_5`), edad, sexo, y el bloque
sociodemográfico (`etnia`, `area`, `educ_madre`, `tipo_nac`) para el conjunto de predictores "D"
(trayectoria + sociodemográfico). No se reconstruyen aquí `mdd`, `insegur`, `hb_madre` ni el resto de
variables del otro proyecto — si se necesitan en el futuro, van en su propio notebook.


In [2]:
cols_per = ['id_upm', 'id_viv', 'id_hogar', 'id_per', 'id_mef', 'fexp', 'estrato',
            'prov', 'area', 'etnia', 'edaddias', 'f1_s1_2', 'dcronica', 'dcronica2_5',
            'nivins_mef']
per_todos, meta_per = pyreadstat.read_dta(F('f1_personas'), usecols=cols_per)

# madres: se construye ANTES de filtrar a niños, porque las madres son otras filas del hogar
# (f1_personas trae a TODOS los integrantes del hogar, no solo a los niños)
madres = (per_todos.loc[pd.to_numeric(per_todos.nivins_mef, errors='coerce').notna(),
                        ['id_upm', 'id_viv', 'id_hogar', 'id_mef', 'nivins_mef']]
                    .rename(columns={'nivins_mef': 'educ_madre'}))

per = per_todos[per_todos.dcronica.notna()].copy()   # restringe a niños con estado nutricional (0-59m)
per['edad_m'] = pd.to_numeric(per.edaddias, errors='coerce') / 30.4375
per['sexo'] = pd.to_numeric(per.f1_s1_2, errors='coerce').map({1: 'M', 2: 'F'})

# el id_mef del niño viene vacío en f1_personas; el puente es f2_salud_ninez
puente, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=K + ['id_mef'])

nin = (per.drop(columns=['id_mef', 'nivins_mef'])
          .merge(puente, on=K, how='left', validate='1:1')
          .merge(madres, on=['id_upm', 'id_viv', 'id_hogar', 'id_mef'],
                 how='left', validate='m:1'))

print('nin:', nin.shape)
print('\ncobertura de variables clave:')
print(cobertura(nin, ['educ_madre', 'etnia', 'area', 'sexo', 'dcronica2_5']).to_string())
assert len(nin) == 22331, f'esperados 22.331 registros de f1_personas, hay {len(nin)}'


nin: (22331, 17)

cobertura de variables clave:
dcronica2_5    0.627
educ_madre     0.964
area           1.000
etnia          1.000
sexo           1.000


In [3]:
# ── peso al nacer y tipo de nacimiento (para el conjunto sociodemográfico D) ──
cols_nac = K + ['f2_s4d_432', 'f2_s4d_433', 'f2_s4d_443_b', 'f2_s4d_444', 'f2_s4d_446']
sn_nac, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=cols_nac)

sn_nac['peso_nacer'] = rango(sn_nac.f2_s4d_443_b, 500, 6000).fillna(rango(sn_nac.f2_s4d_444, 500, 6000))
sn_nac['sem_gest'] = rango(sn_nac.f2_s4d_432, 24, 45)
sn_nac['prematuro'] = (sn_nac.sem_gest < 37).astype(float).where(sn_nac.sem_gest.notna())

# percentil 10 interno de peso por semana de gestación, como aproximación a PEG
# (no es un estándar tipo INTERGROWTH-21st: sirve para comparar grupos dentro de ENDI)
p10 = sn_nac.groupby('sem_gest').peso_nacer.transform(lambda s: s.quantile(.10))
sn_nac['peg'] = (sn_nac.peso_nacer < p10).astype(float).where(sn_nac.peso_nacer.notna())

sn_nac['tipo_nac'] = np.select(
    [(sn_nac.prematuro == 1) & (sn_nac.peg == 1),
     (sn_nac.prematuro == 1) & (sn_nac.peg == 0),
     (sn_nac.prematuro == 0) & (sn_nac.peg == 1)],
    ['prematuro_peg', 'prematuro_aeg', 'termino_peg'], default='termino_aeg')
sn_nac.loc[sn_nac.peso_nacer.isna() | sn_nac.sem_gest.isna(), 'tipo_nac'] = None

nin = nin.merge(sn_nac[K + ['tipo_nac']], on=K, how='left', validate='1:1')
print('cobertura tipo_nac:', nin.tipo_nac.notna().mean().round(3))

nin.to_parquet(RAIZ / 'cache_nin_iccs.parquet')


cobertura tipo_nac: 0.753


## 3. Registros longitudinales del carné de control

Reconstrucción del bloque de transcripción del carné (hasta 12 mediciones por niño) y cálculo del
puntaje Z de talla para la edad. Cada paso de filtrado se reporta explícitamente — este es el embudo
completo que el revisor pidió documentar.


In [4]:
piezas = ['edad', 'peso', 'talla']
cols_carne = (K + ['f2_s4f_462', 'f2_s4f_463_a', 'f2_s4f_463_b']
              + [f'f2_s4f_464_{p}_{i}' for i in range(1, 13) for p in piezas])
tr, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=cols_carne)

largo = []
for i in range(1, 13):
    d = tr[K].copy()
    d['orden'] = i
    for p in piezas:
        d[p] = pd.to_numeric(tr[f'f2_s4f_464_{p}_{i}'], errors='coerce')
    largo.append(d)
largo = pd.concat(largo, ignore_index=True)

n0 = len(largo.dropna(subset=['talla']))
print(f'[1] mediciones con talla registrada: {n0:,}')

L = largo.copy()
L['edad'] = L.edad.where(L.edad.between(0, 60))
L['peso'] = L.peso.where(L.peso.between(1.5, 30))
L['talla'] = L.talla.where(L.talla.between(40, 125))
L = L.dropna(subset=['talla']).dropna(subset=['edad', 'talla'])
print(f'[2] tras filtro de rango físico plausible: {len(L):,}  (-{n0 - len(L):,})')


[1] mediciones con talla registrada: 71,403
[2] tras filtro de rango físico plausible: 70,900  (-503)


In [5]:
# LMS de la OMS (talla/longitud para la edad), ya construido en un notebook previo de preparación
lms = (pd.read_csv(RAIZ / "lms_talla_edad_oms.csv")
         .sort_values(['sexo', 'mes'])
         .drop_duplicates(['sexo', 'mes'], keep='last'))
assert len(lms) == 122, f'la tabla LMS debe tener 122 filas (2 sexos x 61 meses), tiene {len(lms)}'

n2 = len(L)
L2 = L.merge(nin[K + ['sexo']], on=K, how='left', validate='m:1')
L2['mes'] = L2.edad.round().astype(int).clip(0, 60)
L2 = L2.merge(lms, on=['sexo', 'mes'], how='left', validate='m:1')
L2['z'] = ((L2.talla / L2.M) ** L2.L - 1) / (L2.L * L2.S)

n3 = L2.z.notna().sum()
print(f'[3] con z calculable (sexo + LMS disponibles): {n3:,}  (-{n2 - n3:,})')

# criterio OMS: descartar |Z| > 6 como error de medición, no como caso real
L2['z_ok'] = L2.z.where(L2.z.between(-6, 6))
n4 = L2.z_ok.notna().sum()
print(f'[4] dentro de |Z| <= 6: {n4:,}  (-{n3 - n4:,})')


[3] con z calculable (sexo + LMS disponibles): 69,931  (-969)
[4] dentro de |Z| <= 6: 69,754  (-177)


## 4. Medición estandarizada de ENDI (patrón de referencia)

Promedio de las tomas repetidas del levantamiento antropométrico propio de la encuesta (no el carné),
usada como patrón de referencia para evaluar la calidad del carné.


In [6]:
cols_ant = K + ['f1_s5_5_1', 'f1_s5_5_2', 'f1_s5_5_3',   # longitud (decúbito)
                'f1_s5_6_1', 'f1_s5_6_2', 'f1_s5_6_3']  # talla (de pie)
ant, _ = pyreadstat.read_dta(F('f1_personas'), usecols=cols_ant)

for c in cols_ant[len(K):]:
    ant[c] = rango(ant[c], 40, 125)

lon = ant[['f1_s5_5_1', 'f1_s5_5_2', 'f1_s5_5_3']]
tal = ant[['f1_s5_6_1', 'f1_s5_6_2', 'f1_s5_6_3']]
ant['medida'] = lon.mean(axis=1).fillna(tal.mean(axis=1))

ant = ant.merge(nin[K + ['sexo', 'edad_m']], on=K, how='inner', validate='1:1')
ant['mes'] = ant.edad_m.round().astype(int).clip(0, 60)
ant = ant.merge(lms, on=['sexo', 'mes'], how='left', validate='m:1')
ant['z_endi'] = ((ant.medida / ant.M) ** ant.L - 1) / (ant.L * ant.S)

n_endi_0 = ant.z_endi.notna().sum()
ant['z_endi'] = ant.z_endi.where(ant.z_endi.between(-6, 6))
n_endi_1 = ant.z_endi.notna().sum()
print(f'ENDI con z calculable: {n_endi_0:,} -> dentro de |Z|<=6: {n_endi_1:,}  (-{n_endi_0 - n_endi_1:,})')
print(f'\n% z_endi < -2 (DCI 0-59m): {(ant.z_endi < -2).mean():.1%}  (oficial INEC: 17,5%)')


ENDI con z calculable: 22,331 -> dentro de |Z|<=6: 22,330  (-1)

% z_endi < -2 (DCI 0-59m): 18.0%  (oficial INEC: 17,5%)


## 5. Calidad del carné frente al patrón de referencia

Comparación pareada: última medición del carné dentro de un margen de ±3 meses respecto a la medición
ENDI del mismo niño. Este es el análisis que sustenta la **Tabla 4** del artículo (n = 4.761, todas las
edades) — describe la calidad general del registro de rutina, no la corrección que se aplica al modelo.


In [7]:
ult = (L2.dropna(subset=['z_ok']).sort_values('edad')
         .groupby(K).tail(1)[K + ['edad', 'z_ok']]
         .rename(columns={'edad': 'edad_carne', 'z_ok': 'z_carne'}))

comp = ant[K + ['z_endi', 'edad_m']].merge(ult, on=K, how='inner')
comp['dif_edad'] = comp.edad_m - comp.edad_carne
comp = comp[comp.dif_edad.between(-1, 3)].dropna(subset=['z_endi', 'z_carne']).copy()
comp['dif'] = comp.z_carne - comp.z_endi

print(f'n (comparación pareada, <3 meses de diferencia) = {len(comp):,}')
assert len(comp) == 4761, f'se esperaban 4.761 pares, hay {len(comp)} — revisar criterios de filtrado'

sesgo_global = comp.dif.mean()
print(f'\nsesgo global (carné - ENDI): {sesgo_global:+.4f} DE')
print(f'desviación de la diferencia:  {comp.dif.std():.4f} DE')
print(f'correlación: {comp.z_endi.corr(comp.z_carne):.3f}')

comp['z_carne_corr'] = comp.z_carne - sesgo_global  # corrección por el sesgo global (Tabla 4)

resumen_sens_espec = {}
for nom, col in [('sin corregir', 'z_carne'), ('corregido (sesgo global)', 'z_carne_corr')]:
    d = (comp[col] < -2).astype(int)
    e = (comp.z_endi < -2).astype(int)
    sens = ((d == 1) & (e == 1)).sum() / (e == 1).sum()
    espec = ((d == 0) & (e == 0)).sum() / (e == 0).sum()
    resumen_sens_espec[nom] = (sens, espec)
    print(f'\n{nom}: sensibilidad {sens:.1%} | especificidad {espec:.1%}')


n (comparación pareada, <3 meses de diferencia) = 4,761

sesgo global (carné - ENDI): +0.3876 DE
desviación de la diferencia:  0.7546 DE
correlación: 0.799

sin corregir: sensibilidad 55.3% | especificidad 97.1%

corregido (sesgo global): sensibilidad 75.9% | especificidad 90.0%


## 6. Estabilidad del sesgo por edad y por nivel de talla (respuesta al comentario 23)

El sesgo del carné **no es constante**. Se prueba explícitamente si varía por tramo de edad y por
nivel de talla (afectados vs. no afectados), con una prueba de tendencia en cada caso.


In [8]:
tramos = [0, 12, 24, 36, 48, 60]
etiquetas = ['0-11', '12-23', '24-35', '36-47', '48-59']
comp['tramo_edad'] = pd.cut(comp.edad_carne, tramos, labels=etiquetas, right=False)

por_edad = comp.groupby('tramo_edad', observed=True).dif.agg(['mean', 'std', 'count']).round(3)
print('sesgo medio por tramo de edad:')
print(por_edad.to_string())

# prueba de tendencia: ¿el sesgo cae con la edad? (regresión simple dif ~ edad_carne)
pend, intercepto, r, p_tend_edad, se = stats.linregress(comp.edad_carne, comp.dif)
print(f'\npendiente = {pend:.4f} DE/mes | p = {p_tend_edad:.2e}')


sesgo medio por tramo de edad:
             mean    std  count
tramo_edad                     
0-11        0.495  0.850   2614
12-23       0.294  0.649   1359
24-35       0.216  0.514    496
36-47       0.171  0.421    212
48-59       0.092  0.334     80

pendiente = -0.0121 DE/mes | p = 1.65e-36


In [9]:
comp['afectado'] = comp.z_endi < -2
por_talla = comp.groupby('afectado').dif.agg(['mean', 'std', 'count']).round(3)
print('sesgo medio por nivel de talla (según patrón de referencia):')
print(por_talla.to_string())

t, p_talla = stats.ttest_ind(comp.loc[comp.afectado, 'dif'],
                              comp.loc[~comp.afectado, 'dif'], equal_var=False)
print(f'\ndiferencia entre grupos: t = {t:.2f} | p = {p_talla:.2e}')


sesgo medio por nivel de talla (según patrón de referencia):
           mean    std  count
afectado                     
False     0.356  0.769   3819
True      0.516  0.677    942

diferencia entre grupos: t = 6.30 | p = 3.77e-10


## 7. Sesgo específico para el rango que usa el modelo (≤ 18 meses)

**Este es el número que se aplica para corregir las trayectorias del modelo — no el sesgo global de
la sección 5.** Se calcula igual que el sesgo global, pero restringido a las mediciones del carné con
edad ≤ 18 meses, porque esa es la única ventana que alimenta al modelo.


In [10]:
comp_18 = comp[comp.edad_carne <= 18]
sesgo_18m = comp_18.dif.mean()
print(f'n (pares con medición de carné <=18 meses) = {len(comp_18):,}')
print(f'sesgo específico <=18 meses: {sesgo_18m:+.4f} DE')
print(f'(vs. sesgo global de toda la muestra: {sesgo_global:+.4f} DE)')

SESGO_MODELO = sesgo_18m  # <- se usa de aquí en adelante para construir las variables de trayectoria


n (pares con medición de carné <=18 meses) = 3,584
sesgo específico <=18 meses: +0.4504 DE
(vs. sesgo global de toda la muestra: +0.3876 DE)


## 8. Variables de trayectoria (con la corrección de sesgo correcta)

In [11]:
Z = L2.dropna(subset=['z_ok']).copy()
Z['z'] = Z.z_ok - SESGO_MODELO
Z = Z[Z.edad <= 18].sort_values(K + ['edad'])
print(f'mediciones <=18m corregidas: {len(Z):,} | niños: {Z.groupby(K).ngroups:,}')


def rasgos(g):
    g = g.sort_values('edad')
    z, e = g.z.values, g.edad.values
    d = {'n_med': len(z), 'edad_pri': e[0], 'edad_ult': e[-1],
         'z_pri': z[0], 'z_ult': z[-1], 'z_min': z.min(),
         'z_medio': z.mean(), 'delta': z[-1] - z[0]}
    if len(z) >= 3 and np.ptp(e) >= 3:
        d['pend'] = np.polyfit(e, z, 1)[0]
    else:
        d['pend'] = np.nan
    d['bajo_alguna'] = float((z < -2).any())
    return pd.Series(d)


R = Z.groupby(K).apply(rasgos, include_groups=False).reset_index()
print(f'niños con rasgos de trayectoria: {len(R):,}')


mediciones <=18m corregidas: 58,090 | niños: 11,268
niños con rasgos de trayectoria: 11,268


In [12]:
blanco = nin[(nin.edad_m >= 24) & (nin.edad_m <= 42) & nin.dcronica2_5.notna()][
    K + ['dcronica2_5', 'edad_m', 'prov', 'etnia', 'area', 'educ_madre', 'tipo_nac', 'sexo']]
blanco['y'] = pd.to_numeric(blanco.dcronica2_5, errors='coerce')

# prevalencia en la ventana 24-42 meses (para contexto — no es la prevalencia nacional 24-59m)
print(f'prevalencia DCI en 24-42 meses: {blanco.y.mean():.1%}')

M = R.merge(blanco, on=K, how='inner')
M = M[M.n_med >= 3].reset_index(drop=True)
print(f'\nn = {len(M):,} | prevalencia en la muestra de modelado = {M.y.mean():.3f}')
assert len(M) == 2057, f'se esperaban 2.057 niños en la muestra de modelado, hay {len(M)}'


prevalencia DCI en 24-42 meses: 20.4%

n = 2,057 | prevalencia en la muestra de modelado = 0.226


## 8b. Estadística descriptiva de la muestra de modelado (Tabla 1 del artículo)

Respuesta al comentario 31: tabla descriptiva de la muestra antes de presentar resultados de modelos.


In [13]:
etnia_map = {1: 'Indígena', 2: 'Afroecuatoriana', 3: 'Montubia', 4: 'Mestiza', 5: 'Blanca'}
area_map = {1: 'Urbana', 2: 'Rural'}

print(f'n = {len(M):,}')
print(f"\nSexo:\n{M.sexo.value_counts(normalize=True, dropna=False).round(3).to_string()}")
print(f'\nEdad al momento del resultado (24-42m), media (DE): '
      f'{M.edad_m.mean():.1f} ({M.edad_m.std():.1f}) meses')
print(f"\nÁrea:\n{M.area.map(area_map).value_counts(normalize=True, dropna=False).round(3).to_string()}")
print(f"\nEtnia:\n{M.etnia.map(etnia_map).value_counts(normalize=True, dropna=False).round(3).to_string()}")
print(f'\nNúmero de mediciones previas a los 18m, media (DE): {M.n_med.mean():.1f} ({M.n_med.std():.1f})')
print(f'\nPrevalencia de DCI (24-42m): {M.y.mean():.1%}')


n = 2,057

Sexo:
sexo
M    0.509
F    0.491

Edad al momento del resultado (24-42m), media (DE): 31.9 (5.2) meses

Área:
area
Urbana    0.523
Rural     0.477

Etnia:
etnia
Mestiza            0.817
Indígena           0.124
Afroecuatoriana    0.031
Montubia           0.020
Blanca             0.007

Número de mediciones previas a los 18m, media (DE): 6.9 (3.1)

Prevalencia de DCI (24-42m): 22.6%


## 9. Verificación de fuga de información (respuesta al comentario 22)

¿Los niños usados para calibrar el sesgo (sección 5, n = 4.761) se superponen con los usados para
entrenar el modelo (sección 8, n = 2.057)? Si es así, ¿el sesgo cambia de forma relevante al excluir
la superposición?


In [14]:
ids_validacion = set(map(tuple, comp[K].values))
ids_modelo = set(map(tuple, M[K].values))
interseccion = ids_validacion & ids_modelo

print(f'Superposición: {len(interseccion)} de {len(ids_modelo)} niños del modelo '
      f'({len(interseccion) / len(ids_modelo):.1%}) también están en la muestra de calibración del sesgo.')

comp_sin_solapo = comp[~comp[K].apply(tuple, axis=1).isin(interseccion)]
sesgo_sin_solapo = comp_sin_solapo.dif.mean()
print(f'\nsesgo global excluyendo la superposición: {sesgo_sin_solapo:+.4f} DE (n={len(comp_sin_solapo):,})')
print(f'sesgo global con la muestra completa:        {sesgo_global:+.4f} DE (n={len(comp):,})')
print(f'diferencia: {abs(sesgo_sin_solapo - sesgo_global):.4f} DE — sin efecto práctico' )


Superposición: 364 de 2057 niños del modelo (17.7%) también están en la muestra de calibración del sesgo.

sesgo global excluyendo la superposición: +0.4015 DE (n=4,397)
sesgo global con la muestra completa:        +0.3876 DE (n=4,761)
diferencia: 0.0138 DE — sin efecto práctico


## 10. Comparación de conjuntos de predictores y algoritmos

Cuatro conjuntos de predictores (A: sociodemográfico solo, B: una medición, C: trayectoria completa,
D: trayectoria + sociodemográfico), y sobre el conjunto C, tres algoritmos de clasificación.


In [15]:
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, brier_score_loss

RASGOS = ['z_ult', 'z_medio', 'z_min', 'z_pri', 'delta', 'pend', 'n_med', 'edad_ult', 'bajo_alguna']

M['etnia_c'] = M.etnia.map({1: 'indigena', 2: 'afro', 3: 'montubia', 4: 'mestiza', 5: 'blanca'})
M['area_c'] = M.area.map({1: 'urbano', 2: 'rural'})
M['educ_c'] = M.educ_madre.map({1: 'basica', 2: 'media', 3: 'superior'})
M['tipo_c'] = M.tipo_nac.fillna('sin_dato')
socio = ['etnia_c', 'area_c', 'educ_c', 'tipo_c']

conjuntos = {
    'A. Solo sociodemográfico': ([], socio),
    'B. Una medición (z_ult)': (['z_ult', 'edad_ult'], []),
    'C. Trayectoria completa': (RASGOS, []),
    'D. Trayectoria + sociodemográfico': (RASGOS, socio),
}

# misma partición agrupada por UPM para las cuatro comparaciones (comparabilidad entre conjuntos)
cv_grupo = GroupKFold(5)

y_, g = M.y.astype(int), M.id_upm
resultados_conjuntos = {}
for nombre, (num, cat) in conjuntos.items():
    pasos = []
    if num:
        pasos.append(('n', Pipeline([('i', SimpleImputer(strategy='median')),
                                      ('s', StandardScaler())]), num))
    if cat:
        pasos.append(('c', OneHotEncoder(handle_unknown='ignore', drop='first'), cat))
    pp = Pipeline([('prep', ColumnTransformer(pasos)),
                   ('clf', LogisticRegression(max_iter=2000))])
    p = cross_val_predict(pp, M, y_, groups=g, cv=cv_grupo, method='predict_proba')[:, 1]
    auc = roc_auc_score(y_, p)
    # AUC por pliegue (media ± DE), para dar una idea de la incertidumbre de muestreo
    # entre conglomerados (respuesta al comentario 34: ¿0,877 y 0,876 son distintos?)
    aucs_pliegue = cross_val_score(pp, M, y_, groups=g, cv=cv_grupo, scoring='roc_auc')
    resultados_conjuntos[nombre] = (p, auc, aucs_pliegue)
    print(f'{nombre:<36} AUC agrupado {auc:.4f}  |  media pliegues {aucs_pliegue.mean():.4f} '
          f'± DE {aucs_pliegue.std():.4f}')


A. Solo sociodemográfico             AUC agrupado 0.6406  |  media pliegues 0.6464 ± DE 0.0338
B. Una medición (z_ult)              AUC agrupado 0.8671  |  media pliegues 0.8684 ± DE 0.0187
C. Trayectoria completa              AUC agrupado 0.8758  |  media pliegues 0.8772 ± DE 0.0161
D. Trayectoria + sociodemográfico    AUC agrupado 0.8750  |  media pliegues 0.8762 ± DE 0.0155


In [16]:
modelos_alt = {
    'Regresión logística': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', LogisticRegression(max_iter=2000))]),
    'Random Forest': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=500, max_depth=5,
                                        min_samples_leaf=20, random_state=42))]),
    'Gradient Boosting (HistGB)': HistGradientBoostingClassifier(
        max_depth=3, max_iter=200, random_state=42),
}

X = M[RASGOS]
for nombre, m in modelos_alt.items():
    p = cross_val_predict(m, X, y_, groups=g, cv=GroupKFold(5), method='predict_proba')[:, 1]
    print(f'{nombre:<28} AUC {roc_auc_score(y_, p):.4f}')


Regresión logística          AUC 0.8764
Random Forest                AUC 0.8754
Gradient Boosting (HistGB)   AUC 0.8588


## 11. Modelo final: matriz de confusión real y calibración (respuesta al comentario 41)

Predicciones reales de validación cruzada agrupada — no una reconstrucción por redondeo.


In [17]:
pp_final = Pipeline([
    ('prep', ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median')),
                        ('s', StandardScaler())]), RASGOS)])),
    ('clf', LogisticRegression(max_iter=2000))])

p_final = cross_val_predict(pp_final, M, y_, groups=g, cv=GroupKFold(5), method='predict_proba')[:, 1]
auc_final = roc_auc_score(y_, p_final)
print(f'AUC (trayectoria completa, regresión logística): {auc_final:.4f}')

d = pd.DataFrame({'p': p_final, 'y': y_.values})
print('\ncalibración por decil:')
print(d.groupby(pd.qcut(d.p, 10, labels=False)).agg(
      pred=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size')).round(3).to_string())

# puntaje de Brier (respuesta al comentario 28: el AUC solo no alcanza para un problema de salud)
brier = brier_score_loss(y_, p_final)
print(f'\npuntaje de Brier: {brier:.4f}  (0 = predicción perfecta; referencia ingenua = '
      f'{((y_ - y_.mean())**2).mean():.4f})')


AUC (trayectoria completa, regresión logística): 0.8758

calibración por decil:
    pred    obs    n
p                   
0  0.005  0.005  206
1  0.016  0.024  206
2  0.030  0.029  205
3  0.053  0.058  206
4  0.086  0.078  206
5  0.134  0.161  205
6  0.225  0.189  206
7  0.368  0.385  205
8  0.534  0.519  206
9  0.801  0.811  206

puntaje de Brier: 0.1102  (0 = predicción perfecta; referencia ingenua = 0.1750)


In [18]:
UMBRAL = 0.20
marca = d.p >= UMBRAL

vp = int(((marca) & (d.y == 1)).sum())
fp = int(((marca) & (d.y == 0)).sum())
fn = int(((~marca) & (d.y == 1)).sum())
vn = int(((~marca) & (d.y == 0)).sum())

sens = vp / (vp + fn)
espec = vn / (vn + fp)
vpp = vp / (vp + fp)

print(f'Matriz de confusión real (p >= {UMBRAL}):')
print(f'  VP = {vp} | FP = {fp}')
print(f'  FN = {fn} | VN = {vn}')
print(f'\nsensibilidad = {sens:.1%} | especificidad = {espec:.1%} | VPP = {vpp:.1%}')
print(f'niños marcados: {marca.mean():.1%}')


Matriz de confusión real (p >= 0.2):
  VP = 379 | FP = 375
  FN = 86 | VN = 1217

sensibilidad = 81.5% | especificidad = 76.4% | VPP = 50.3%
niños marcados: 36.7%


## 12. Entrenamiento final y guardado del modelo

In [19]:
modelo_final = Pipeline([
    ('prep', ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median')),
                        ('s', StandardScaler())]), RASGOS)])),
    ('clf', LogisticRegression(max_iter=2000))])

modelo_final.fit(M[RASGOS], M.y.astype(int))

joblib.dump({
    'modelo': modelo_final,
    'rasgos': RASGOS,
    'sesgo_carne_modelo': SESGO_MODELO,   # sesgo <=18m, el que corrige las mediciones de entrada
    'sesgo_carne_global': sesgo_global,   # sesgo global, solo para referencia/documentación
    'lms': lms,
    'auc_cv': round(auc_final, 4),
    'n_entren': len(M),
    'umbral_recomendado': UMBRAL,
}, RAIZ / 'modelo_riesgo_dci.joblib')

print('modelo guardado |', len(M), 'casos |', f'{M.y.mean():.3f}', 'prevalencia |',
      f'sesgo aplicado: {SESGO_MODELO:+.4f} DE')


modelo guardado | 2057 casos | 0.226 prevalencia | sesgo aplicado: +0.4504 DE


## 13. Función de inferencia

In [20]:
def riesgo_dci(mediciones, sexo, ruta=RAIZ / 'modelo_riesgo_dci.joblib'):
    """mediciones: lista de (edad_meses, talla_cm), con edad_meses <= 18.
    sexo: 'M' o 'F'.
    Devuelve la probabilidad estimada de desnutrición crónica infantil (DCI) entre los 24 y 42 meses.
    """
    b = joblib.load(ruta)
    t = b['lms'][b['lms'].sexo == sexo].set_index('mes')

    z, e = [], []
    for edad, talla in sorted(mediciones):
        if not (0 <= edad <= 18 and 40 <= talla <= 125):
            continue
        r = t.loc[int(round(min(edad, 60)))]
        z_val = ((talla / r.M) ** r.L - 1) / (r.L * r.S) - b['sesgo_carne_modelo']
        z.append(z_val)
        e.append(edad)

    if len(z) < 3:
        return None
    z, e = np.array(z), np.array(e)

    f = {'z_ult': z[-1], 'z_medio': z.mean(), 'z_min': z.min(), 'z_pri': z[0],
         'delta': z[-1] - z[0], 'n_med': len(z), 'edad_ult': e[-1],
         'bajo_alguna': float((z < -2).any()),
         'pend': np.polyfit(e, z, 1)[0] if np.ptp(e) >= 3 else np.nan}

    return float(b['modelo'].predict_proba(pd.DataFrame([f])[b['rasgos']])[0, 1])


# prueba con dos trayectorias contrastantes
sano = [(2, 57.5), (6, 67.0), (12, 75.5), (17, 81.0)]
riesgo = [(2, 54.0), (6, 62.0), (12, 68.0), (17, 71.5)]
print(f'trayectoria normal:      {riesgo_dci(sano, "M"):.1%}')
print(f'trayectoria descendente: {riesgo_dci(riesgo, "M"):.1%}')


trayectoria normal:      3.4%
trayectoria descendente: 95.1%


## 14. Resumen de cifras para el artículo

Ejecutar al final y copiar los valores impresos directamente al texto — evita transcribir a mano y
que un número quede desactualizado respecto al código que realmente lo produjo.


In [21]:
print('=== CIFRAS PARA EL ARTÍCULO ===\n')
print(f'Muestra de calibración del sesgo (Tabla 4):          n = {len(comp):,}')
print(f'  Sensibilidad sin corregir:                          {resumen_sens_espec["sin corregir"][0]:.1%}')
print(f'  Especificidad sin corregir:                         {resumen_sens_espec["sin corregir"][1]:.1%}')
print(f'  Sensibilidad corregida (sesgo global):              {resumen_sens_espec["corregido (sesgo global)"][0]:.1%}')
print(f'  Especificidad corregida (sesgo global):             {resumen_sens_espec["corregido (sesgo global)"][1]:.1%}')
print(f'  Sesgo global (carné - ENDI):                        {sesgo_global:+.3f} DE')
print(f'  Error aleatorio (DE de la diferencia):               {comp.dif.std():.3f} DE')
print(f'  Sesgo específico <=18m (usado en el modelo):        {SESGO_MODELO:+.3f} DE')
print(f'  Superposición con muestra de modelado:              {len(interseccion)} de {len(ids_modelo)} ({len(interseccion)/len(ids_modelo):.1%})')
print()
print(f'Muestra de modelado:                                  n = {len(M):,}')
print(f'  Prevalencia (24-42 meses):                          {M.y.mean():.1%}')
print(f'  AUC sociodemográfico solo:                          {resultados_conjuntos["A. Solo sociodemográfico"][1]:.4f}')
print(f'  AUC una medición:                                   {resultados_conjuntos["B. Una medición (z_ult)"][1]:.4f}')
print(f'  AUC trayectoria completa:                           {resultados_conjuntos["C. Trayectoria completa"][1]:.4f} '
      f'(media pliegues {resultados_conjuntos["C. Trayectoria completa"][2].mean():.4f} '
      f'± DE {resultados_conjuntos["C. Trayectoria completa"][2].std():.4f})')
print(f'  AUC trayectoria + sociodemográfico:                 {resultados_conjuntos["D. Trayectoria + sociodemográfico"][1]:.4f} '
      f'(media pliegues {resultados_conjuntos["D. Trayectoria + sociodemográfico"][2].mean():.4f} '
      f'± DE {resultados_conjuntos["D. Trayectoria + sociodemográfico"][2].std():.4f})')
print(f'  Puntaje de Brier (trayectoria completa):            {brier:.4f}')
print()
print(f'Matriz de confusión real (p>=0.20):                   VP={vp} FP={fp} FN={fn} VN={vn}')
print(f'  Sensibilidad: {sens:.1%} | Especificidad: {espec:.1%} | VPP: {vpp:.1%}')


=== CIFRAS PARA EL ARTÍCULO ===

Muestra de calibración del sesgo (Tabla 4):          n = 4,761
  Sensibilidad sin corregir:                          55.3%
  Especificidad sin corregir:                         97.1%
  Sensibilidad corregida (sesgo global):              75.9%
  Especificidad corregida (sesgo global):             90.0%
  Sesgo global (carné - ENDI):                        +0.388 DE
  Error aleatorio (DE de la diferencia):               0.755 DE
  Sesgo específico <=18m (usado en el modelo):        +0.450 DE
  Superposición con muestra de modelado:              364 de 2057 (17.7%)

Muestra de modelado:                                  n = 2,057
  Prevalencia (24-42 meses):                          22.6%
  AUC sociodemográfico solo:                          0.6406
  AUC una medición:                                   0.8671
  AUC trayectoria completa:                           0.8758 (media pliegues 0.8772 ± DE 0.0161)
  AUC trayectoria + sociodemográfico:                 